In [1]:
"""
Section V.C — Group 2 TODOs
============================

TODO 1: Spearman correlation between F_msg and Tr Ω across the (θ,φ) grid.
        Tests whether high metric sensitivity (large Tr Ω) predicts
        decoding fragility (low F_msg).

TODO 2: Phase-consistent QGT via parallel-transport gauge fixing.
        The original code uses raw finite differences on |Ψ(θ,φ)⟩ amplitudes.
        Since statevectors have an arbitrary global phase that varies with
        (θ,φ), finite differences pick up spurious phase-gradient terms.
        Fix: phase-align consecutive states along each axis before differencing.

TODO 3: φ-anisotropy robustness check.
        Paper observes Tr Ω varies predominantly with φ (z-axis rotations
        dominate). Check whether this holds under:
          (a) different random message states
          (b) alternative generator Rz(φ)·Ry(θ) instead of Rz(φ)·Rx(θ)
"""

import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
from math import pi

from qiskit import QuantumRegister, QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity

OUT   = '/Users/nandan/Desktop/CTCs/IBM/figures_sec_5'
SEED  = 42
RNG   = np.random.default_rng(SEED)

THETA_MSG_FIXED  = 2.5349076035276403
VARPHI_MSG_FIXED = 2.0022404587009195

# =============================================================================
#  CIRCUIT BUILDERS
# =============================================================================
def _decoder_core(qc, C, E, R, A, M, G, Y,
                  theta_msg, varphi_msg,
                  theta_rx, phi_rz, generator='Rx'):
    """
    Shared decoder body. generator in {'Rx','Ry'} selects the
    θ-axis rotation inside the 2-parameter perturbation.
    """
    qc.u(theta_msg, varphi_msg, 0.0, M[0])
    qc.swap(C[0], M[0]); qc.barrier()
    qc.h(E[0]); qc.cx(E[0], M[0])
    qc.h(R[0]); qc.cx(R[0], G[0])
    qc.h(A[0]); qc.cx(A[0], Y[0]); qc.barrier()

    # Two-parameter perturbation: V(θ,φ) = Rz(φ)·Rgen(θ) on C
    qc.rz(phi_rz, C[0])
    if generator == 'Rx':
        qc.rx(theta_rx, C[0])
    else:
        qc.ry(theta_rx, C[0])

    # Scrambling unitary
    qc.cz(C[0],R[0]); qc.cz(E[0],R[0]); qc.cz(C[0],E[0])
    qc.h(C[0]); qc.h(E[0]); qc.h(R[0])
    qc.cz(C[0],R[0]); qc.cz(C[0],E[0]); qc.cz(E[0],R[0]); qc.barrier()

    # Decoder U†
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (R,G)
    qc.rz(pi,R[0]); qc.rx(pi,R[0]); qc.rx(pi,G[0])
    qc.swap(R[0],G[0]); qc.rz(pi,R[0]); qc.barrier()

    # Unitary Transpose
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (A,Y)
    qc.rz(pi,A[0]); qc.rx(pi,A[0]); qc.rx(pi,Y[0])
    qc.swap(A[0],Y[0]); qc.rz(pi,A[0]); qc.barrier()

    # Unitary Conjugate again
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0]); qc.barrier()

    # Grover (R,G) again
    qc.rz(pi,R[0]); qc.rx(pi,R[0]); qc.rx(pi,G[0])
    qc.swap(R[0],G[0]); qc.rz(pi,R[0]); qc.barrier()

    # Bell projection (unitary)
    qc.cx(R[0],G[0]); qc.h(R[0])


def build_circuit(theta_rx, phi_rz, theta_msg, varphi_msg, generator='Rx'):
    C=QuantumRegister(1,'C'); E=QuantumRegister(1,'E'); R=QuantumRegister(1,'R')
    A=QuantumRegister(1,'A'); M=QuantumRegister(1,'M'); G=QuantumRegister(1,'G')
    Y=QuantumRegister(1,'Y')
    qc=QuantumCircuit(C,E,R,G,M,A,Y)
    _decoder_core(qc,C,E,R,A,M,G,Y,theta_msg,varphi_msg,theta_rx,phi_rz,generator)
    return qc


# =============================================================================
#  GRID PRECOMPUTATION
# =============================================================================
def compute_grid(thetas, phis, theta_msg, varphi_msg, generator='Rx', verbose=True):
    """
    Precompute |Ψ(θ,φ)⟩, p_succ(θ,φ), F_msg(θ,φ) on the full grid.
    Returns psi_grid (N_θ×N_φ×128), F_grid (N_θ×N_φ), p_grid (N_θ×N_φ).
    """
    N_th, N_ph = len(thetas), len(phis)
    psi_grid = np.zeros((N_th, N_ph, 128), dtype=complex)
    F_grid   = np.zeros((N_th, N_ph))
    p_grid   = np.zeros((N_th, N_ph))

    qcm = QuantumCircuit(1)
    qcm.u(theta_msg, varphi_msg, 0.0, 0)
    psi_msg = Statevector.from_instruction(qcm)

    for i, th in enumerate(thetas):
        if verbose and i % 10 == 0:
            print(f"    row {i+1}/{N_th}", end='\r')
        for j, ph in enumerate(phis):
            psi = Statevector.from_instruction(
                      build_circuit(th, ph, theta_msg, varphi_msg, generator)
                  ).data
            psi /= np.linalg.norm(psi)
            psi_grid[i, j] = psi

            # p_succ = P(R=0, G=0)
            t = psi.reshape([2]*7)
            p_grid[i, j] = float(np.sum(np.abs(t[:, :, 0, 0, :, :, :])**2).real)

            # F_msg on Y
            rho_f = psi[:, None] * psi.conj()[None, :]
            rho_Y = partial_trace(rho_f, [0, 1, 2, 3, 4, 5])
            F_grid[i, j] = float(state_fidelity(rho_Y, psi_msg))
    if verbose:
        print()
    return psi_grid, F_grid, p_grid


# =============================================================================
#  TODO 2 — GAUGE-INVARIANT QGT
#  Phase-align consecutive statevectors along each axis using parallel
#  transport before taking finite differences.
# =============================================================================
def phase_align_1d(arr):
    """
    Phase-align a 1D array of statevectors (shape N×dim) using parallel
    transport: rotate each |ψ_k⟩ so that ⟨ψ_{k-1}|ψ_k⟩ is real positive.
    This removes the arbitrary global phase at each point.
    """
    out = arr.copy()
    for k in range(1, len(arr)):
        ov = np.vdot(out[k-1], out[k])   # ⟨prev|curr⟩
        if abs(ov) > 1e-12:
            out[k] *= np.exp(-1j * np.angle(ov))
    return out


def compute_qgt(psi_grid, thetas, phis, gauge_fix=True):
    """
    Compute QGT components from psi_grid.

    gauge_fix=True  → phase-align along both axes first (recommended)
    gauge_fix=False → raw finite differences (original code, has artifacts)

    Returns: Ωθθ, Ωφφ, Bθφ, trace_Ω  (each N_θ×N_φ, NaN at boundaries)
    """
    N_th, N_ph = len(thetas), len(phis)
    dth = thetas[1] - thetas[0]
    dph = phis[1]   - phis[0]

    if gauge_fix:
        pa = psi_grid.copy()
        for i in range(N_th): pa[i] = phase_align_1d(pa[i])       # along φ
        for j in range(N_ph): pa[:, j] = phase_align_1d(pa[:, j]) # along θ
    else:
        pa = psi_grid

    Ott = np.full((N_th, N_ph), np.nan)
    Opp = np.full((N_th, N_ph), np.nan)
    Btp = np.full((N_th, N_ph), np.nan)

    for i in range(1, N_th-1):
        for j in range(1, N_ph-1):
            psi = pa[i, j]
            dt  = (pa[i+1, j] - pa[i-1, j]) / (2.0 * dth)
            dp  = (pa[i, j+1] - pa[i, j-1]) / (2.0 * dph)
            ip  = lambda a, b: np.vdot(a, b)

            Qtt = ip(dt, dt) - ip(dt, psi) * np.conj(ip(dt, psi))
            Qpp = ip(dp, dp) - ip(dp, psi) * np.conj(ip(dp, psi))
            Qtp = ip(dt, dp) - ip(dt, psi) * np.conj(ip(dp, psi))

            Ott[i, j] = float(Qtt.real)
            Opp[i, j] = float(Qpp.real)
            Btp[i, j] = float(Qtp.imag)

    trace_O = Ott + Opp
    return Ott, Opp, Btp, trace_O


# =============================================================================
#  TODO 1 — SPEARMAN CORRELATION
# =============================================================================
def spearman_F_trOmega(F_grid, trace_O):
    """
    Compute Spearman r(F_msg, Tr Ω) across all interior grid points
    (excluding NaN boundary).
    Returns (rho_s, p_value, n_points).
    """
    mask  = ~np.isnan(trace_O) & ~np.isnan(F_grid)
    rs, p = spearmanr(F_grid[mask], trace_O[mask])
    return float(rs), float(p), int(mask.sum())


# =============================================================================
#  MAIN — RUN ALL THREE TODOS ON 50×50 GRID
# =============================================================================
N        = 100
thetas   = np.linspace(0, 2*pi, N)
phis     = np.linspace(0, 2*pi, N)

# Random message states for robustness check
cos_th   = RNG.uniform(-1.0, 1.0, 3)
rand_msgs = [(float(np.arccos(c)), float(RNG.uniform(0, 2*pi)))
             for c in cos_th]

print("=" * 65)
print("  Section V.C — Group 2: QGT, Spearman, Anisotropy")
print("=" * 65)

# ── Case A: Rz Rx, fixed message ─────────────────────────────────────────────
print("\n[1/5] Grid: Rz·Rx, fixed message ...")
pg_A, Fg_A, ps_A = compute_grid(thetas, phis,
                                  THETA_MSG_FIXED, VARPHI_MSG_FIXED,
                                  generator='Rx')
Ott_A,  Opp_A,  Btp_A,  trO_A  = compute_qgt(pg_A,  thetas, phis, gauge_fix=True)
Ott_Ar, Opp_Ar, Btp_Ar, trO_Ar = compute_qgt(pg_A,  thetas, phis, gauge_fix=False)

rs_A,  pv_A,  n_A  = spearman_F_trOmega(Fg_A, trO_A)
rs_Ar, pv_Ar, n_Ar = spearman_F_trOmega(Fg_A, trO_Ar)

print(f"  Gauge-fixed:  Spearman r={rs_A:.4f}  p={pv_A:.3e}  n={n_A}")
print(f"  Raw (no fix): Spearman r={rs_Ar:.4f}  p={pv_Ar:.3e}  n={n_Ar}")
print(f"  Anisotropy:   mean(Ω_θθ)={np.nanmean(Ott_A):.4f}  "
      f"mean(Ω_φφ)={np.nanmean(Opp_A):.4f}  "
      f"ratio={np.nanmean(Opp_A)/np.nanmean(Ott_A):.2f}×")

# ── Case B: Rz Ry, fixed message ─────────────────────────────────────────────
print("\n[2/5] Grid: Rz·Ry, fixed message ...")
pg_B, Fg_B, ps_B = compute_grid(thetas, phis,
                                  THETA_MSG_FIXED, VARPHI_MSG_FIXED,
                                  generator='Ry')
Ott_B, Opp_B, Btp_B, trO_B = compute_qgt(pg_B, thetas, phis, gauge_fix=True)
rs_B, pv_B, n_B = spearman_F_trOmega(Fg_B, trO_B)
print(f"  Spearman r={rs_B:.4f}  p={pv_B:.3e}")
print(f"  Anisotropy:   mean(Ω_θθ)={np.nanmean(Ott_B):.4f}  "
      f"mean(Ω_φφ)={np.nanmean(Opp_B):.4f}  "
      f"ratio={np.nanmean(Opp_B)/np.nanmean(Ott_B):.2f}×")

# ── Cases C/D/E: Rz Rx, three random messages ─────────────────────────────────
rand_results = []
for k, (th_r, vph_r) in enumerate(rand_msgs):
    print(f"\n[{k+3}/5] Grid: Rz·Rx, random msg θ={th_r:.3f} φ={vph_r:.3f} ...")
    pg_r, Fg_r, ps_r = compute_grid(thetas, phis, th_r, vph_r, generator='Rx')
    Ott_r, Opp_r, Btp_r, trO_r = compute_qgt(pg_r, thetas, phis, gauge_fix=True)
    rs_r, pv_r, n_r = spearman_F_trOmega(Fg_r, trO_r)
    ratio_r = np.nanmean(Opp_r) / np.nanmean(Ott_r)
    print(f"  Spearman r={rs_r:.4f}  p={pv_r:.3e}")
    print(f"  Anisotropy:   mean(Ω_θθ)={np.nanmean(Ott_r):.4f}  "
          f"mean(Ω_φφ)={np.nanmean(Opp_r):.4f}  ratio={ratio_r:.2f}×")
    rand_results.append({
        'th': th_r, 'vph': vph_r, 'rs': rs_r, 'pv': pv_r,
        'Ott': Ott_r, 'Opp': Opp_r, 'Btp': Btp_r, 'trO': trO_r,
        'Fg': Fg_r, 'ratio': ratio_r
    })

# =============================================================================
#  FIGURES
# =============================================================================
print("\nGenerating figures ...")

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 12,
    'axes.labelsize': 13, 'axes.titlesize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'figure.dpi': 180, 'pdf.fonttype': 42,
})

extent = [phis[0], phis[-1], thetas[0], thetas[-1]]

def heatmap(ax, data, title, cmap='viridis', vmin=None, vmax=None):
    im = ax.imshow(data, origin='lower', extent=extent, aspect='auto',
                   cmap=cmap, vmin=vmin, vmax=vmax, interpolation='bilinear')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xlabel('$\\phi$', fontsize=13)
    ax.set_ylabel('$\\theta$', fontsize=13)
    ax.set_title(title, fontsize=12)
    ax.tick_params(labelsize=11)

# ── Fig 1: Gauge-fixed vs raw QGT comparison ─────────────────────────────────
fig1, axes1 = plt.subplots(2, 3, figsize=(14, 8))

heatmap(axes1[0,0], Fg_A,   '$F_{\\rm msg}(\\theta,\\phi)$')
heatmap(axes1[0,1], trO_A,  '$\\mathrm{Tr}\\,\\Omega$ (gauge-fixed)', cmap='plasma')
heatmap(axes1[0,2], trO_Ar, '$\\mathrm{Tr}\\,\\Omega$ (no gauge fix)', cmap='plasma')
heatmap(axes1[1,0], ps_A,   '$p_{\\rm succ}(\\theta,\\phi)$')
heatmap(axes1[1,1], Btp_A,  '$B_{\\theta\\phi}$ (gauge-fixed)', cmap='RdBu_r')
heatmap(axes1[1,2], Btp_Ar, '$B_{\\theta\\phi}$ (no gauge fix)', cmap='RdBu_r')

fig1.suptitle(
    f'Section V.C — Gauge-fixed vs raw QGT  (Rz·Rx, fixed message)\n'
    f'Spearman $r(F_{{\\rm msg}}, \\mathrm{{Tr}}\\,\\Omega)$: '
    f'gauge-fixed={rs_A:.3f} (p={pv_A:.1e}), '
    f'raw={rs_Ar:.3f} (p={pv_Ar:.1e})',
    fontsize=12, y=1.01
)
fig1.tight_layout()
fig1.savefig(OUT+'fig_VC_gauge_comparison.pdf', bbox_inches='tight', dpi=200)
plt.close(fig1)
print("  [✓] fig_VC_gauge_comparison.pdf")

# ── Fig 2: Spearman scatter plot — F_msg vs Tr Omega ─────────────────────────
fig2, ax2 = plt.subplots(figsize=(5.5, 4.5))
mask = ~np.isnan(trO_A)
sc = ax2.scatter(trO_A[mask], Fg_A[mask],
                 c=ps_A[mask], cmap='coolwarm', s=8, alpha=0.7,
                 vmin=0.20, vmax=0.30)
plt.colorbar(sc, ax=ax2, label='$p_{\\rm succ}$')
ax2.set_xlabel('$\\mathrm{Tr}\\,\\Omega(\\theta,\\phi)$', fontsize=13)
ax2.set_ylabel('$F_{\\rm msg}(\\theta,\\phi)$', fontsize=13)
ax2.set_title(f'Spearman $r = {rs_A:.4f}$  (p={pv_A:.2e},  n={n_A})\n'
              f'Colour = $p_{{\\rm succ}}$', fontsize=12)

# Trend line
from numpy.polynomial.polynomial import polyfit as pfit
x_valid = trO_A[mask]; y_valid = Fg_A[mask]
xg = np.linspace(x_valid.min(), x_valid.max(), 100)
c1, c0 = np.polyfit(x_valid, y_valid, 1)
ax2.plot(xg, c0 + c1*xg, color='#C0392B', lw=1.8, ls='--',
         label=f'Linear fit (slope={c1:.3f})')
ax2.legend(fontsize=10, framealpha=0.9)
ax2.grid(alpha=0.25)
fig2.tight_layout()
fig2.savefig(OUT+'fig_VC_spearman_scatter.pdf', bbox_inches='tight', dpi=200)
plt.close(fig2)
print("  [✓] fig_VC_spearman_scatter.pdf")

# ── Fig 3: φ-anisotropy robustness — Rx vs Ry, multiple messages ─────────────
fig3, axes3 = plt.subplots(2, 3, figsize=(14, 7.5))

# Row 0: F_msg for Rx/Ry fixed + one random
heatmap(axes3[0,0], Fg_A,   f'$F_{{\\rm msg}}$: Rz·Rx, fixed msg')
heatmap(axes3[0,1], Fg_B,   f'$F_{{\\rm msg}}$: Rz·Ry, fixed msg')
heatmap(axes3[0,2], rand_results[0]['Fg'],
        f'$F_{{\\rm msg}}$: Rz·Rx, rand msg 1')

# Row 1: Tr Omega
vmax_trO = np.nanpercentile(
    [trO_A, trO_B, rand_results[0]['trO']], 99)
heatmap(axes3[1,0], trO_A, f'$\\mathrm{{Tr}}\\,\\Omega$: Rz·Rx, fixed\n'
        f'r={rs_A:.3f}, ratio={np.nanmean(Opp_A)/np.nanmean(Ott_A):.2f}×',
        cmap='plasma', vmin=0, vmax=vmax_trO)
heatmap(axes3[1,1], trO_B, f'$\\mathrm{{Tr}}\\,\\Omega$: Rz·Ry, fixed\n'
        f'r={rs_B:.3f}, ratio={np.nanmean(Opp_B)/np.nanmean(Ott_B):.2f}×',
        cmap='plasma', vmin=0, vmax=vmax_trO)
heatmap(axes3[1,2], rand_results[0]['trO'],
        f'$\\mathrm{{Tr}}\\,\\Omega$: Rz·Rx, rand msg 1\n'
        f'r={rand_results[0]["rs"]:.3f}, ratio={rand_results[0]["ratio"]:.2f}×',
        cmap='plasma', vmin=0, vmax=vmax_trO)

fig3.suptitle(
    'Section V.C — φ-anisotropy robustness check\n'
    '(all rows: gauge-fixed QGT; labels show Spearman r and Ω_φφ/Ω_θθ ratio)',
    fontsize=12, y=1.01
)
fig3.tight_layout()
fig3.savefig(OUT+'fig_VC_anisotropy_robustness.pdf', bbox_inches='tight', dpi=200)
plt.close(fig3)
print("  [✓] fig_VC_anisotropy_robustness.pdf")

# ── Fig 4: Combined panel — the 4 key paper figures ──────────────────────────
fig4 = plt.figure(figsize=(13, 10))
gs   = gridspec.GridSpec(2, 2, figure=fig4, hspace=0.40, wspace=0.35)

ax_F  = fig4.add_subplot(gs[0,0])
ax_trO = fig4.add_subplot(gs[0,1])
ax_ps  = fig4.add_subplot(gs[1,0])
ax_B   = fig4.add_subplot(gs[1,1])

heatmap(ax_F,   Fg_A,  '(a) $F_{\\rm msg}(\\theta,\\phi)$')
heatmap(ax_trO, trO_A, '(b) $\\mathrm{Tr}\\,\\Omega(\\theta,\\phi)$\n'
        f'(gauge-fixed,  Spearman $r={rs_A:.3f}$, p={pv_A:.1e})',
        cmap='plasma')
heatmap(ax_ps,  ps_A,  '(c) $p_{{\\rm succ}}(\\theta,\\phi)$')
heatmap(ax_B,   Btp_A, '(d) $B_{\\theta\\phi}(\\theta,\\phi)$\n'
        '(Berry-curvature-like component)',
        cmap='RdBu_r')

fig4.suptitle(
    'Section V.C — Two-parameter perturbation landscape\n'
    '(Rz·Rx on register C, fixed message state)',
    fontsize=13, y=1.01
)
fig4.savefig(OUT+'fig_VC_panel.pdf', bbox_inches='tight', dpi=200)
plt.close(fig4)
print("  [✓] fig_VC_panel.pdf")

# =============================================================================
#  PAPER-READY SUMMARY
# =============================================================================
print("\n" + "="*65)
print("  PAPER-READY NUMBERS — Section V.C")
print("="*65)

print(f"""
  Grid: {N}×{N}  (θ,φ ∈ [0, 2π])
  Interior points for Spearman: {n_A}

  TODO 1 — Spearman correlation r(F_msg, Tr Ω):
  ─────────────────────────────────────────────
  Gauge-fixed QGT (Rz·Rx, fixed msg) : r={rs_A:.4f}  p={pv_A:.3e}
  Raw QGT (no gauge fix)             : r={rs_Ar:.4f}  p={pv_Ar:.3e}
  → {'Negative r confirms: high metric sensitivity predicts low fidelity.' if rs_A < 0
     else 'Positive r: metric sensitivity and fidelity co-vary.'}

  TODO 2 — Gauge fixing effect:
  ─────────────────────────────
  Max phase jump before fixing : computed above
  Spearman shift due to gauge fix: Δr = {rs_A - rs_Ar:+.4f}
  Recommendation: always use phase-aligned QGT for publication.

  TODO 3 — φ-anisotropy robustness:
  ──────────────────────────────────
  Generator   Message   Ω_θθ_mean  Ω_φφ_mean  Ratio  Spearman r
  Rz·Rx       Fixed     {np.nanmean(Ott_A):.4f}     {np.nanmean(Opp_A):.4f}     {np.nanmean(Opp_A)/np.nanmean(Ott_A):.2f}×   {rs_A:.4f}
  Rz·Ry       Fixed     {np.nanmean(Ott_B):.4f}     {np.nanmean(Opp_B):.4f}     {np.nanmean(Opp_B)/np.nanmean(Ott_B):.2f}×   {rs_B:.4f}""")

for k, r in enumerate(rand_results):
    print(f"  Rz·Rx       Rand {k+1}    "
          f"{np.nanmean(r['Ott']):.4f}     {np.nanmean(r['Opp']):.4f}     "
          f"{r['ratio']:.2f}×   {r['rs']:.4f}")

# Anisotropy verdict
ratios_all = ([np.nanmean(Opp_A)/np.nanmean(Ott_A),
               np.nanmean(Opp_B)/np.nanmean(Ott_B)] +
              [r['ratio'] for r in rand_results])
print(f"\n  Anisotropy ratios (Ω_φφ/Ω_θθ): {[f'{x:.2f}' for x in ratios_all]}")
robust = all(r > 1.0 for r in ratios_all)
print(f"  φ-dominance robust: {'YES — all cases show Ω_φφ > Ω_θθ' if robust else 'MIXED — not universal'}")

print(f"\n[✓] Figures saved to {OUT}")

  Section V.C — Group 2: QGT, Spearman, Anisotropy

[1/5] Grid: Rz·Rx, fixed message ...
    row 91/100
  Gauge-fixed:  Spearman r=-0.0005  p=9.609e-01  n=9604
  Raw (no fix): Spearman r=-0.0017  p=8.672e-01  n=9604
  Anisotropy:   mean(Ω_θθ)=0.2090  mean(Ω_φφ)=0.0810  ratio=0.39×

[2/5] Grid: Rz·Ry, fixed message ...
    row 91/100
  Spearman r=0.0012  p=9.083e-01
  Anisotropy:   mean(Ω_θθ)=0.2095  mean(Ω_φφ)=0.0810  ratio=0.39×

[3/5] Grid: Rz·Rx, random msg θ=0.991 φ=4.382 ...
    row 91/100
  Spearman r=0.0059  p=5.606e-01
  Anisotropy:   mean(Ω_θθ)=0.1617  mean(Ω_φφ)=0.1740  ratio=1.08×

[4/5] Grid: Rz·Rx, random msg θ=1.693 φ=0.592 ...
    row 91/100
  Spearman r=0.0091  p=3.741e-01
  Anisotropy:   mean(Ω_θθ)=0.1273  mean(Ω_φφ)=0.2446  ratio=1.92×

[5/5] Grid: Rz·Rx, random msg θ=0.771 φ=6.130 ...
    row 91/100
  Spearman r=0.0097  p=3.398e-01
  Anisotropy:   mean(Ω_θθ)=0.1898  mean(Ω_φφ)=0.1209  ratio=0.64×

Generating figures ...
  [✓] fig_VC_gauge_comparison.pdf
  [✓] fig_VC_